# Kalman filter: notebook

The running example of the Note: a robot commanded 0.5 m per second along a corridor (slip variance Q = 0.01 m²), located by a ceiling beacon (reading variance R = 0.16 m²), first guess 0 m with variance 1 m². We check every number in the Note and run the experiments behind its figures.

In [1]:
import numpy as np
import pandas as pd

import numpy as np

U, Q, R = 0.5, 0.01, 0.16
X0, P0 = 0.0, 1.0
X_TRUE = np.array([0.3, 0.73, 1.22, 1.88, 2.45, 2.79, 3.28, 3.72, 4.24, 4.58, 5.1])
Z = np.array([0.82, 1.85, 2.01, 2.65, 2.19, 4.18, 2.95, 4.68, 4.45, 4.75])


def kf1d(z=Z, u=U, q=Q, r=R, x=X0, p=P0):
    """The one-dimensional Kalman filter. Returns one dict per step with the prediction, gain and update."""
    rows = []
    for zk in z:
        xb, pb = x + u, p + q                     # predict
        k = pb / (pb + r)                         # gain
        x, p = xb + k * (zk - xb), (1 - k) * pb   # update
        rows.append(dict(x_pred=xb, p_pred=pb, K=k, z=zk, x=x, p=p))
    return rows


# the two-variable version: state (position, speed), no odometry, position readings only
F = np.array([[1.0, 1.0], [0.0, 1.0]])
H = np.array([[1.0, 0.0]])
Q2 = np.diag([0.01, 0.001])


def kf2d(z=Z, Hm=H, x=None, P=None):
    """The matrix Kalman filter with the constant-speed model. Returns one dict per step."""
    x = np.zeros(2) if x is None else x
    P = np.eye(2) if P is None else P
    rows = []
    for zk in z:
        xb, Pb = F @ x, F @ P @ F.T + Q2
        S = (Hm @ Pb @ Hm.T)[0, 0] + R
        K = (Pb @ Hm.T / S).ravel()
        y = zk - (Hm @ xb)[0]
        x, P = xb + K * y, (np.eye(2) - np.outer(K, Hm)) @ Pb
        rows.append(dict(x_pred=xb, P_pred=Pb, S=S, K=K, y=y, x=x, P=P))
    return rows


def normal(x, m, v):
    return np.exp(-0.5 * (x - m) ** 2 / v) / np.sqrt(2 * np.pi * v)

## 1. R from a calibration run (Note, Section 7.1)

The robot parked on a mark at exactly 2.00 m; the beacon reported its position 100 times. The readings are in `data/beacon_calibration.csv`.

In [2]:
cal = pd.read_csv("data/beacon_calibration.csv")
print(len(cal), "readings, mean", round(cal.reading.mean(), 3), "m, sample variance", round(cal.reading.var(), 3), "m^2")

100 readings, mean 2.018 m, sample variance 0.134 m^2


## 2. The weight that gives the smallest variance (Note, Section 4)

First step: prediction variance 1.01, reading variance 0.16. Search all weights from 0 to 1.

In [3]:
w = np.linspace(0, 1, 100_001)
v = w**2 * R + (1 - w)**2 * (P0 + Q)
print("best w", w[v.argmin()].round(4), " formula", round((P0 + Q) / (P0 + Q + R), 4), " variance", v.min().round(4))

best w 0.8632  formula 0.8632  variance 0.1381


Multiplying the two bell curves gives the same mean and variance.

In [4]:
g = np.linspace(-5, 6, 200_001)
prod = normal(g, 0.5, 1.01) * normal(g, 0.82, R)
prod /= np.trapezoid(prod, g)
m = np.trapezoid(g * prod, g)
print("product: mean", round(m, 3), " variance", round(np.trapezoid((g - m)**2 * prod, g), 4))

product: mean 0.776  variance 0.1381


## 3. Ten steps on the corridor (Note, Section 5)

In [5]:
df = pd.DataFrame(kf1d()).round(4)
df.insert(0, "step", range(1, 11)); df["true"] = X_TRUE[1:]
df

,step,x_pred,p_pred,K,z,x,p,true
0,1,0.5000,1.0100,0.8632,0.82,0.7762,0.1381,0.73
1,2,1.2762,0.1481,0.4807,1.85,1.5521,0.0769,1.22
2,3,2.0521,0.0869,0.3520,2.01,2.0373,0.0563,1.88
3,4,2.5373,0.0663,0.2930,2.65,2.5703,0.0469,2.45
4,5,3.0703,0.0569,0.2623,2.19,2.8394,0.0420,2.79
5,6,3.3394,0.0520,0.2452,4.18,3.5455,0.0392,3.28
6,7,4.0455,0.0492,0.2353,2.95,3.7877,0.0376,3.72
7,8,4.2877,0.0476,0.2295,4.68,4.3777,0.0367,4.24
8,9,4.8777,0.0467,0.2260,4.45,4.7811,0.0362,4.58
9,10,5.2811,0.0462,0.2239,4.75,5.1622,0.0358,5.10


In [6]:
rows = kf1d()
est = np.array([r["x"] for r in rows])
print("RMSE filter", np.sqrt(np.mean((est - X_TRUE[1:])**2)).round(3), " readings", np.sqrt(np.mean((Z - X_TRUE[1:])**2)).round(3),
      " commands only", np.sqrt(np.mean((X0 + U * np.arange(1, 11) - X_TRUE[1:])**2)).round(3))
# steady state: run long
p = P0
for _ in range(200):
    pb = p + Q; k = pb / (pb + R); p = (1 - k) * pb
print("steady-state gain", round(k, 4), " variance", round(p, 4))

RMSE filter 0.171  readings 0.506  commands only 0.271
steady-state gain 0.2207  variance 0.0353


## 4. Position and speed, position readings only (Note, Section 6)

In [7]:
r2 = kf2d()
for i in (0, 1, 9):
    r = r2[i]
    print(f"step {i+1}: x_pred {r['x_pred'].round(3)}  P_pred {r['P_pred'].round(4).tolist()}  S {r['S']:.4f}  K {r['K'].round(4)}"
          f"  y {r['y']:.3f}  x {r['x'].round(3)}  P {r['P'].round(4).tolist()}")
print("true average speed", (X_TRUE[-1] - X_TRUE[0]) / 10)

step 1: x_pred [0. 0.]  P_pred [[2.01, 1.0], [1.0, 1.001]]  S 2.1700  K [0.9263 0.4608]  y 0.820  x [0.76  0.378]  P [[0.1482, 0.0737], [0.0737, 0.5402]]
step 2: x_pred [1.137 0.378]  P_pred [[0.8458, 0.6139], [0.6139, 0.5412]]  S 1.0058  K [0.8409 0.6103]  y 0.713  x [1.737 0.813]  P [[0.1345, 0.0977], [0.0977, 0.1665]]
step 10: x_pred [5.078 0.445]  P_pred [[0.1122, 0.0203], [0.0203, 0.0081]]  S 0.2722  K [0.4121 0.0744]  y -0.328  x [4.943 0.421]  P [[0.0659, 0.0119], [0.0119, 0.0066]]
true average speed 0.48


## 5. Observability (Note, Section 6.4)

In [8]:
for name, Hm in (("position sensor", H), ("speed sensor", np.array([[0.0, 1.0]]))):
    O = np.vstack([Hm, Hm @ F])
    print(name, "O =", O.tolist(), " rank", np.linalg.matrix_rank(O))
print("variance of position after 10 steps: position sensor", round(kf2d()[-1]['P'][0, 0], 3),
      " speed sensor", round(kf2d(z=np.full(10, 0.5), Hm=np.array([[0.0, 1.0]]))[-1]['P'][0, 0], 3))

position sensor O = [[1.0, 0.0], [1.0, 1.0]]  rank 2
speed sensor O = [[0.0, 1.0], [0.0, 1.0]]  rank 1
variance of position after 10 steps: position sensor 0.066  speed sensor 2.683


## 6. Choosing Q (Note, Section 7.2)

2000 simulated runs of 50 steps with the true slip variance 0.01; filters with three values of Q; root-mean-square error over steps 11 to 50.

In [9]:
def sim(n, T=50):
    r = np.random.default_rng(n)
    xt, xs, zs = r.normal(0, 1), [], []
    for _ in range(T):
        xt += 0.5 + r.normal(0, 0.1); xs.append(xt); zs.append(xt + r.normal(0, 0.4))
    return np.array(xs), np.array(zs)
runs = [sim(n) for n in range(2000)]
for q in (1e-4, 1e-2, 1.0):
    e = [(np.array([r["x"] for r in kf1d(z=zs, q=q)]) - xs)[10:] for xs, zs in runs]
    print(f"Q = {q}: RMSE {np.sqrt(np.mean(np.square(e))):.3f} m, final gain {kf1d(q=q)[-1]['K']:.3f}")
print("readings alone:", np.sqrt(np.mean([np.mean((zs - xs)[10:]**2) for xs, zs in runs])).round(3))

Q = 0.0001: RMSE 0.305 m, final gain 0.100
Q = 0.01: RMSE 0.187 m, final gain 0.224
Q = 1.0: RMSE 0.353 m, final gain 0.877
readings alone: 0.399


Only the ratio matters: Q, R and the first variance all ten times larger give the same estimates.

In [10]:
a, b = kf1d(), kf1d(q=0.1, r=1.6, p=10.0)
print("largest difference in the estimates:", max(abs(x["x"] - y["x"]) for x, y in zip(a, b)))

largest difference in the estimates: 0.0
